# How stable are gPS and gps_TA to perturbation of the disease list?

The published pleiotropy metrics are counts over the **1,394 disease terms that happen to have a
qualifying GWAS**. That list is an accident of what has been studied, and it will grow. This notebook
asks the obvious robustness question: if we had happened to be missing 20% of those diseases, would the
manuscript's conclusions look the same?

Each of **100 replicates** drops a uniformly random 20% of disease terms (1,394 → 1,115), recomputes gPS
and gps_TA over the survivors, and re-runs three things:

1. **Rank stability** — Spearman correlation of the subsampled metric against the full-data metric.
2. **Non-linearity with drug approval** — the quadratic logistic fit, its LR test and fitted peak.
3. **Drug-target enrichment, high versus low pleiotropy** — at cuts frozen from the full data, and at
   cuts re-derived inside each replicate.

## Decisions taken before running anything

| Decision | Choice | Why |
| -------- | ------ | --- |
| gps_TA on a cropped disease list | **one TA per disease** via the documented `therapy_area_hierarchy` priority (first match, else `other`) | the only decomposable rule that is faithful to the published column — see below |
| which diseases are dropped | **uniform at random**, 20% | cleanest null; matches "we happen not to have GWAS for these" |
| genetic support | **held fixed** | `score_all` is a max over propagated associations and cannot be decomposed by disease from the pair table; this isolates pleiotropy as the varying quantity |

### Why one-TA-per-disease, and what it costs

The published `uniqueTherapeuticAreas` is a **study**-level union: each contributing study contributes the
set of top-of-ontology areas spanned by *all* of its `diseaseIds`. That is not decomposable by disease —
a disease inherits areas from its study-mates, so dropping a disease would leave its borrowed areas
attached to the survivors and the count would fail to fall when it should. Two decomposable alternatives
were measured against the published column over all 8,285 genes:

| Rule | Exact match | Mean | Max |
| ---- | ----------- | ---- | --- |
| one TA per disease, hierarchy first-match | **86.3%** | 2.42 | 20 |
| every top-level area a disease descends from | 26.8% | 4.06 | 22 |
| *published* | — | 2.53 | 21 |

The multi-area rule inflates by 60% because 627 of the 1,394 disease terms sit under more than one
top-level area. The hierarchy rule is used here, and — importantly — **the 100% baseline is recomputed
with the same rule**, so every number below compares like with like and the subsampling effect is never
confounded with the definitional difference. gPS needs no such treatment: it is a plain count of distinct
disease terms, so the 100% baseline reproduces the published `uniqueDiseases` exactly.

In [1]:
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import chi2

sys.path.insert(0, "../or10-optimism-validation")
from or10_stats import or_rs, support_mask

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"
N_REPLICATES = 100
KEEP_FRACTION = 0.80
SEED = 20260813

## The therapeutic-area map

`therapy_area_hierarchy` is copied verbatim from
`chapters/01-data-preparation/04_qualifying_dataset_generation.ipynb` (cell 4), including its order,
which *is* the priority: the first matching root wins, and a disease matching none is `other`.

In [2]:
# Verbatim from 04_qualifying_dataset_generation.ipynb. Dict order is the priority order.
THERAPY_AREA_HIERARCHY = {
    "EFO_0001444": "measurement",
    "MONDO_0045024": "cancer or benign tumor",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0000540": "immune system disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0003765": "sign or symptom",
}
PRIORITY = [k for k in THERAPY_AREA_HIERARCHY if k != "EFO_0001444"]

disease_index = pd.read_parquet(RELEASE + "output/disease/disease.parquet", columns=["id", "descendants"])
roots = disease_index[disease_index["id"].isin(THERAPY_AREA_HIERARCHY)]
descendants = {r.id: set(list(r.descendants) if r.descendants is not None else []) for r in roots.itertuples()}
print("therapeutic-area roots resolved:", len(descendants), "of", len(THERAPY_AREA_HIERARCHY))
assert len(descendants) == len(THERAPY_AREA_HIERARCHY)

therapeutic-area roots resolved: 23 of 23


In [3]:
import ast

genes_df = pd.read_csv(INTERMEDIATE + "genes_therapeutic_areas.csv")
gene_ids = genes_df["geneId"].tolist()
published_gps = genes_df.set_index("geneId")["uniqueDiseases"]
published_ta = genes_df.set_index("geneId")["uniqueTherapeuticAreas"]

l2g = pd.read_csv(INTERMEDIATE + "l2g_diseases_full-r1.csv", usecols=["geneId", "diseaseIds"])
l2g = l2g[l2g["geneId"].isin(set(gene_ids))]
pairs = (
    l2g.assign(traitId=l2g["diseaseIds"].map(ast.literal_eval))
    .explode("traitId")[["geneId", "traitId"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
disease_terms = np.array(sorted(pairs["traitId"].unique()))
print("genes:", len(gene_ids), "| gene-disease pairs:", len(pairs), "| disease terms:", len(disease_terms))
assert len(gene_ids) == 8285 and len(disease_terms) == 1394


def single_area(term):
    """One therapeutic area per disease: first match in the hierarchy's own order, else `other`."""
    areas = {root for root, kids in descendants.items() if term == root or term in kids}
    for root in PRIORITY:
        if root in areas:
            return root
    return "other"


AREA_OF = {t: single_area(t) for t in disease_terms}
print("distinct areas used:", len(set(AREA_OF.values())))
print("diseases assigned 'other':", sum(1 for v in AREA_OF.values() if v == "other"))

genes: 8285 | gene-disease pairs: 36858 | disease terms: 1394
distinct areas used: 23
diseases assigned 'other': 291


### Vectorise the recomputation

Everything below reduces to two `groupby`-free bincounts over the gene–disease pair list, so a replicate
costs milliseconds and the 100 logistic fits dominate the runtime.

In [4]:
gene_code = pd.Categorical(pairs["geneId"], categories=gene_ids).codes
term_index = {t: i for i, t in enumerate(disease_terms)}
term_code = pairs["traitId"].map(term_index).to_numpy()
area_labels = sorted(set(AREA_OF.values()))
area_index = {a: i for i, a in enumerate(area_labels)}
area_code = np.array([area_index[AREA_OF[t]] for t in disease_terms])
n_genes, n_areas = len(gene_ids), len(area_labels)


def recompute(keep_mask):
    """gPS and gps_TA per gene over the surviving disease terms only."""
    alive = keep_mask[term_code]
    g = gene_code[alive]
    gps = np.bincount(g, minlength=n_genes)  # distinct diseases (pairs are unique)
    seen = np.zeros((n_genes, n_areas), dtype=bool)
    seen[g, area_code[term_code[alive]]] = True
    return gps, seen.sum(axis=1)


full_gps, full_ta = recompute(np.ones(len(disease_terms), dtype=bool))
print("baseline on 100% of diseases (hierarchy rule for TA):")
print(
    f"  gPS    mean {full_gps.mean():.4f} max {full_gps.max()} | published mean {published_gps.mean():.4f} max {published_gps.max()}"
)
print(
    f"  gps_TA mean {full_ta.mean():.4f} max {full_ta.max()} | published mean {published_ta.mean():.4f} max {published_ta.max()}"
)
print(f"  gPS identical to published for {(full_gps == published_gps.loc[gene_ids].to_numpy()).mean():.4f} of genes")
print(f"  gps_TA identical to published for {(full_ta == published_ta.loc[gene_ids].to_numpy()).mean():.4f} of genes")
assert (full_gps == published_gps.loc[gene_ids].to_numpy()).all(), "gPS baseline must reproduce the published count"

baseline on 100% of diseases (hierarchy rule for TA):
  gPS    mean 4.4488 max 148 | published mean 4.4488 max 148
  gps_TA mean 2.4234 max 20 | published mean 2.5308 max 21
  gPS identical to published for 1.0000 of genes
  gps_TA identical to published for 0.8634 of genes


## The drug-target framework

Same pair-level table and the same support definition as everywhere else in `06-review-r1`, imported
from `../or10-optimism-validation/or10_stats.py`. Support is **not** perturbed; only the pleiotropy
metric changes between replicates. A target with no surviving disease association gets pleiotropy 0,
which is exactly how the published framework treats a target carrying no association at all.

In [5]:
chembl = pd.read_parquet(INTERMEDIATE + "ti_pairs_chembl_master-r1.parquet")
chembl["support_all"] = support_mask(chembl).astype(int)
target_row = pd.Series({g: i for i, g in enumerate(gene_ids)})
chembl["gene_row"] = chembl["targetId"].map(target_row)
in_table = chembl["gene_row"].notna().to_numpy()
rows = chembl["gene_row"].fillna(-1).astype(int).to_numpy()
outcome = chembl["approved"].to_numpy()
support = chembl["support_all"].to_numpy()
in_gps = chembl["in_gps"].to_numpy()
print("pairs:", len(chembl), "| approved:", int(outcome.sum()), "| supported:", int(support.sum()))
print("baseline all-GWAS enrichment:", round(or_rs(support_mask(chembl), chembl["approved"])["odds_ratio"], 4))


def pair_metric(per_gene):
    """Spread a per-gene metric onto the pair table, 0 for targets outside the gene table."""
    out = np.zeros(len(chembl))
    out[in_table] = per_gene[rows[in_table]]
    return out


def shape(values):
    """Quadratic logistic fit of approval on log2(metric + 1); returns LR, P, peak, decay point."""
    lx = np.log2(values + 1)
    data = pd.DataFrame({"outcome": outcome, "geneticSupport": support, "lx": lx})
    data["lx2"] = data["lx"] ** 2
    m1 = smf.logit("outcome ~ geneticSupport + lx", data=data).fit(disp=False)
    m2 = smf.logit("outcome ~ geneticSupport + lx + lx2", data=data).fit(disp=False)
    lr = 2 * (float(m2.llf) - float(m1.llf))
    b, a = float(m2.params["lx"]), float(m2.params["lx2"])
    x_peak = -b / (2 * a)
    return {
        "lr_quadratic": lr,
        "p_quadratic": float(chi2.sf(lr, 1)),
        "coef_log2": a,
        "peak": float(2.0**x_peak - 1),
        "decay_point": float(2.0 ** (2 * x_peak) - 1),
    }


def contrast(values, low_max, high_min):
    """Low versus high pleiotropy among supported pairs, against the no-support pairs as reference."""
    supported = (support == 1) & in_gps
    low = supported & (values <= low_max)
    high = supported & (values >= high_min)
    gap = supported & ~(low | high)
    e = np.where(low, 2, np.where(high, 1, 0))
    data = pd.DataFrame({"outcome": outcome, "E": e})[~gap]
    if min(int(data.loc[data["E"] == 2, "outcome"].sum()), int(data.loc[data["E"] == 1, "outcome"].sum())) < 5:
        return None
    fit = smf.logit("outcome ~ C(E)", data=data).fit(disp=False)
    c = np.zeros(len(fit.params))
    c[1], c[2] = -1, 1
    return {
        "or_low": float(np.exp(fit.params.iloc[2])),
        "or_high": float(np.exp(fit.params.iloc[1])),
        "ratio_low_over_high": float(np.exp(fit.params.iloc[2] - fit.params.iloc[1])),
        "p_difference": float(np.ravel(fit.t_test(c).pvalue)[0]),
        "n_low": int((e == 2).sum()),
        "n_high": int((e == 1).sum()),
        "n_approved_low": int(data.loc[data["E"] == 2, "outcome"].sum()),
        "n_approved_high": int(data.loc[data["E"] == 1, "outcome"].sum()),
    }

pairs: 37377 | approved: 4564 | supported: 742
baseline all-GWAS enrichment: 3.6186


### Frozen cuts, derived once from the 100% baseline

The stability question is about the *estimate*, so the primary contrast holds the criterion fixed at
whatever the full data implies, using the same peak / decay-point rule as
`../effective-independent-traits/02_drug_targets.ipynb`: low = M ≤ round(peak), high = M ≥ ⌈decay⌉.
Per-replicate re-derived cuts are also recorded, which lets the two sources of variation be separated.

In [6]:
METRICS = ["gps", "gps_TA"]
baseline = {}
for name, values in [("gps", full_gps), ("gps_TA", full_ta)]:
    s = shape(pair_metric(values))
    low_max, high_min = int(round(s["peak"])), int(np.ceil(s["decay_point"]))
    c = contrast(pair_metric(values), low_max, high_min)
    baseline[name] = {"metric": name, **s, "low_max": low_max, "high_min": high_min, **c}
baseline_df = pd.DataFrame(baseline).T.reset_index(drop=True)
print(
    baseline_df[
        [
            "metric",
            "lr_quadratic",
            "p_quadratic",
            "peak",
            "decay_point",
            "low_max",
            "high_min",
            "or_low",
            "or_high",
            "ratio_low_over_high",
            "p_difference",
        ]
    ]
    .astype({"low_max": int, "high_min": int})
    .round(4)
    .to_string(index=False)
)
FROZEN = {name: (baseline[name]["low_max"], baseline[name]["high_min"]) for name in METRICS}

metric lr_quadratic p_quadratic      peak decay_point  low_max  high_min    or_low   or_high ratio_low_over_high p_difference
   gps    54.171875         0.0  3.694997   21.042995        4        22  4.873011  2.626843            1.855082     0.009824
gps_TA    57.435805         0.0  1.905817    7.443774        2         8  4.209402  2.725043             1.54471     0.071755


## 100 replicates

In [7]:
rng = np.random.default_rng(SEED)
n_keep = int(round(KEEP_FRACTION * len(disease_terms)))
print(f"keeping {n_keep} of {len(disease_terms)} disease terms per replicate, {N_REPLICATES} replicates")

records = []
for r in range(N_REPLICATES):
    keep = np.zeros(len(disease_terms), dtype=bool)
    keep[rng.choice(len(disease_terms), size=n_keep, replace=False)] = True
    sub_gps, sub_ta = recompute(keep)
    alive = sub_gps > 0
    for name, sub, full in [("gps", sub_gps, full_gps), ("gps_TA", sub_ta, full_ta)]:
        values = pair_metric(sub)
        s = shape(values)
        low_max, high_min = FROZEN[name]
        frozen = contrast(values, low_max, high_min)
        d_low, d_high = int(round(s["peak"])), int(np.ceil(s["decay_point"]))
        derived = contrast(values, d_low, d_high)
        row = {
            "replicate": r,
            "metric": name,
            "n_terms_kept": int(keep.sum()),
            "genes_with_no_disease": int((~alive).sum()),
            "mean_metric": float(sub[alive].mean()),
            "max_metric": int(sub.max()),
            "spearman_vs_full": float(stats.spearmanr(full[alive], sub[alive]).statistic),
            "pearson_log2_vs_full": float(stats.pearsonr(np.log2(full[alive] + 1), np.log2(sub[alive] + 1)).statistic),
            **{f"shape_{k}": v for k, v in s.items()},
            "derived_low_max": d_low,
            "derived_high_min": d_high,
        }
        row.update({f"frozen_{k}": v for k, v in (frozen or {}).items()})
        row.update({f"derived_{k}": v for k, v in (derived or {}).items()})
        records.append(row)
    if (r + 1) % 25 == 0:
        print(f"  {r + 1}/{N_REPLICATES} replicates done")

replicates = pd.DataFrame(records)
replicates.to_csv(INTERMEDIATE + "subsample_disease_replicates-r1.csv", index=False)
print("replicate table:", replicates.shape)

keeping 1115 of 1394 disease terms per replicate, 100 replicates


  25/100 replicates done


  50/100 replicates done


  75/100 replicates done


  100/100 replicates done
replicate table: (200, 31)


## Stability summary

In [8]:
def interval(series):
    return pd.Series(
        {
            "median": series.median(),
            "pct2.5": series.quantile(0.025),
            "pct97.5": series.quantile(0.975),
            "min": series.min(),
            "max": series.max(),
        }
    )


summary_rows = []
for name in METRICS:
    block = replicates[replicates["metric"] == name]
    base = baseline[name]
    for label, column, base_value in [
        ("spearman_vs_full", "spearman_vs_full", 1.0),
        ("mean_metric", "mean_metric", float(np.mean(full_gps[full_gps > 0]) if name == "gps" else np.mean(full_ta))),
        ("quadratic_LR", "shape_lr_quadratic", base["lr_quadratic"]),
        ("fitted_peak", "shape_peak", base["peak"]),
        ("frozen_ratio_low_over_high", "frozen_ratio_low_over_high", base["ratio_low_over_high"]),
        ("frozen_or_low", "frozen_or_low", base["or_low"]),
        ("frozen_or_high", "frozen_or_high", base["or_high"]),
        ("derived_ratio_low_over_high", "derived_ratio_low_over_high", base["ratio_low_over_high"]),
    ]:
        stats_row = interval(block[column].dropna())
        summary_rows.append(
            {
                "metric": name,
                "quantity": label,
                "baseline": base_value,
                **stats_row.to_dict(),
                "n_replicates": int(block[column].notna().sum()),
            }
        )
summary = pd.DataFrame(summary_rows)
summary.to_csv(INTERMEDIATE + "subsample_disease_summary-r1.csv", index=False)
print(summary.round(4).to_string(index=False))

metric                    quantity  baseline  median  pct2.5  pct97.5     min     max  n_replicates
   gps            spearman_vs_full    1.0000  0.9499  0.9304   0.9672  0.9266  0.9705           100
   gps                 mean_metric    4.4488  3.8584  3.7153   3.9981  3.6545  4.0182           100
   gps                quadratic_LR   54.1719 50.7741 28.7191  67.3347 22.0574 73.0838           100
   gps                 fitted_peak    3.6950  3.2041  2.8370   3.4998  2.6439  3.6187           100
   gps  frozen_ratio_low_over_high    1.8551  2.4724  1.8039   3.0400  1.5023  3.1341           100
   gps               frozen_or_low    4.8730  4.7722  4.4039   5.2100  4.2796  5.3752           100
   gps              frozen_or_high    2.6268  1.8988  1.6257   2.6830  1.5206  2.9668           100
   gps derived_ratio_low_over_high    1.8551  1.6396  1.3230   2.2076  1.2479  2.4643           100
gps_TA            spearman_vs_full    1.0000  0.9362  0.9019   0.9576  0.8974  0.9629           100


In [9]:
verdict_rows = []
for name in METRICS:
    block = replicates[replicates["metric"] == name]
    base = baseline[name]
    verdict_rows.append(
        {
            "metric": name,
            "baseline_quadratic_P": base["p_quadratic"],
            "replicates_quadratic_P_lt_05": int((block["shape_p_quadratic"] < 0.05).sum()),
            "replicates_quadratic_P_lt_001": int((block["shape_p_quadratic"] < 0.001).sum()),
            "baseline_ratio": base["ratio_low_over_high"],
            "replicates_ratio_gt_1_frozen": int((block["frozen_ratio_low_over_high"] > 1).sum()),
            "replicates_frozen_P_lt_05": int((block["frozen_p_difference"] < 0.05).sum()),
            "replicates_ratio_gt_1_derived": int((block["derived_ratio_low_over_high"] > 1).sum()),
            "replicates_derived_P_lt_05": int((block["derived_p_difference"] < 0.05).sum()),
            "min_spearman_vs_full": block["spearman_vs_full"].min(),
            "replicates_spearman_gt_095": int((block["spearman_vs_full"] > 0.95).sum()),
            "mean_genes_losing_all_diseases": block["genes_with_no_disease"].mean(),
            "n_replicates": len(block),
        }
    )
verdict = pd.DataFrame(verdict_rows)
verdict.to_csv(INTERMEDIATE + "subsample_disease_verdict-r1.csv", index=False)
print(verdict.T.to_string())

                                       0         1
metric                               gps    gps_TA
baseline_quadratic_P                 0.0       0.0
replicates_quadratic_P_lt_05         100       100
replicates_quadratic_P_lt_001        100       100
baseline_ratio                  1.855082   1.54471
replicates_ratio_gt_1_frozen         100       100
replicates_frozen_P_lt_05             98        34
replicates_ratio_gt_1_derived        100       100
replicates_derived_P_lt_05            65        71
min_spearman_vs_full            0.926615  0.897425
replicates_spearman_gt_095            50        15
mean_genes_losing_all_diseases    639.66    639.66
n_replicates                         100       100


In [10]:
# Cut stability: how often does the re-derived criterion land on the frozen one?
cutcount = (
    replicates.groupby(["metric", "derived_low_max", "derived_high_min"]).size().rename("replicates").reset_index()
)
cutcount.to_csv(INTERMEDIATE + "subsample_disease_cut_frequency-r1.csv", index=False)
for name in METRICS:
    print(f"--- {name}: frozen cut <= {FROZEN[name][0]} / >= {FROZEN[name][1]} ---")
    print(cutcount[cutcount["metric"] == name].sort_values("replicates", ascending=False).to_string(index=False))
    print()

--- gps: frozen cut <= 4 / >= 22 ---
metric  derived_low_max  derived_high_min  replicates
   gps                3                17          26
   gps                3                18          24
   gps                3                16          17
   gps                3                19          15
   gps                3                15          10
   gps                3                14           2
   gps                3                20           2
   gps                4                21           2
   gps                3                13           1
   gps                4                20           1

--- gps_TA: frozen cut <= 2 / >= 8 ---
metric  derived_low_max  derived_high_min  replicates
gps_TA                2                 7          61
gps_TA                2                 8          22
gps_TA                2                 6          15
gps_TA                1                 5           1
gps_TA                1                 6           1



## Exports

| File | Contents |
| ---- | -------- |
| `subsample_disease_replicates-r1.csv` | one row per replicate × metric — every statistic computed |
| `subsample_disease_summary-r1.csv` | median and 2.5–97.5 percentile interval per quantity, against the baseline |
| `subsample_disease_verdict-r1.csv` | how many of the 100 replicates preserve each conclusion |
| `subsample_disease_cut_frequency-r1.csv` | which low/high cuts the derivation rule picks across replicates |

Interpretation is in the README.